## Silver Transformation

Reads raw data from `gtfs_bronze` and writes cleaned, typed and normalized tables to `gtfs_silver`.

### What it does
- Casts columns to correct types (DOUBLE, INT, DATE)
- Converts `arrival_time` and `departure_time` from `HH:MM:SS` string to seconds from midnight (handles GTFS times > 24h)
- Adds `route_type_desc` label from `route_type` integer code (GTFS spec)
- Flags invalid stops (`is_valid`) based on Italy bounding box
- Deduplicates rows on natural keys using `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ingestion_ts DESC)`

### Source
`gtfs_bronze.stops`, `gtfs_bronze.routes`, `gtfs_bronze.trips`, `gtfs_bronze.stop_times`, `gtfs_bronze.calendar`, `gtfs_bronze.calendar_dates`

### Target
`gtfs_silver.stops`, `gtfs_silver.routes`, `gtfs_silver.trips`, `gtfs_silver.stop_times`, `gtfs_silver.calendar`, `gtfs_silver.calendar_dates`

### Notes
- Roma has no `calendar.txt` — services defined entirely via `calendar_dates` with `exception_type = 1`
- Milano `calendar.txt` exists but is empty — same applies
- Only Torino uses `exception_type = 2` (service removed)

### Stops

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.stops AS
SELECT stop_id,
    stop_code,
    stop_name,
    stop_desc,
    CAST(stop_lat AS double) as stop_lat,
    CAST(stop_lon AS double) as stop_lon,
    zone_id,
    stop_url,
    COALESCE(CAST(location_type AS int), 0) AS location_type,
    parent_station,
    stop_timezone,
    wheelchair_boarding,
    city,
    ingestion_ts,
    source_file,
    CASE 
        WHEN CAST(stop_lat AS DOUBLE) BETWEEN 36.0 AND 47.5 
        AND CAST(stop_lon AS DOUBLE) BETWEEN 6.5  AND 18.5 THEN TRUE ELSE FALSE
    END AS is_valid
FROM gtfs_bronze.stops
QUALIFY ROW_NUMBER() OVER (PARTITION BY stop_id, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, COUNT(*) as total, SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) as valid
FROM gtfs_silver.stops
GROUP BY city


### Routes  

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.routes AS
SELECT route_id,
    agency_id,
    route_short_name,
    route_long_name,
    CAST(route_type AS int) AS route_type,
    route_url,
    route_color,
    route_text_color,
    route_desc,
    x_accessibilita_linea,
    x_ordinamento_linea,
    route_sort_order,
    city,
    ingestion_ts,
    source_file,
    CASE 
        WHEN CAST(route_type AS int) = 0 THEN 'tram'
        WHEN CAST(route_type AS int) = 1 THEN 'metro'
        WHEN CAST(route_type AS int) = 2 THEN 'rail'
        WHEN CAST(route_type AS int) = 3 THEN 'bus'
        WHEN CAST(route_type AS int) = 4 THEN 'ferry'
        WHEN CAST(route_type AS INT) = 7 THEN 'funicular'
    END AS route_type_desc
FROM gtfs_bronze.routes
QUALIFY ROW_NUMBER() OVER (PARTITION BY route_id, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, route_type_desc, COUNT(*) as total
FROM gtfs_silver.routes
GROUP BY city, route_type_desc
ORDER BY city, total DESC


### Trips

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.trips AS
SELECT route_id,
    service_id,
    trip_id,
    trip_headsign,
    COALESCE(CAST(direction_id AS int),0) AS direction_id,
    shape_id,
    city,
    ingestion_ts,
    source_file
FROM gtfs_bronze.trips
QUALIFY ROW_NUMBER() OVER (PARTITION BY trip_id, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, COUNT(*) as total
FROM gtfs_silver.trips
GROUP BY city
ORDER BY city

### Stop Times

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.stop_times AS
SELECT trip_id,
    stop_id,
    CAST(stop_sequence AS int) AS stop_sequence,
    CAST(SPLIT(arrival_time, ':')[0] AS INT) * 3600
    + CAST(SPLIT(arrival_time, ':')[1] AS INT) * 60
    + CAST(SPLIT(arrival_time, ':')[2] AS INT)
    AS arrival_secs,
    CAST(SPLIT(departure_time, ':')[0] AS INT) * 3600
    + CAST(SPLIT(departure_time, ':')[1] AS INT) * 60
    + CAST(SPLIT(departure_time, ':')[2] AS INT)
    AS departure_secs,
    city,
    ingestion_ts
FROM gtfs_bronze.stop_times
QUALIFY ROW_NUMBER() OVER (PARTITION BY trip_id, stop_id, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, COUNT(*) as total
FROM gtfs_silver.stop_times
GROUP BY city
ORDER BY city


### Calendar

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.calendar AS
SELECT service_id,
    CAST(monday AS int) AS monday,
    CAST(tuesday AS int) AS tuesday,
    CAST(wednesday AS int) AS wednesday,
    CAST(thursday AS int) AS thursday,
    CAST(friday AS int) AS friday,
    CAST(saturday AS int) AS saturday,
    CAST(sunday AS int) AS sunday,
    city,
    ingestion_ts,
    TO_DATE(start_date, 'yyyyMMdd') AS start_date,
    TO_DATE(end_date, 'yyyyMMdd') AS end_date
FROM gtfs_bronze.calendar
QUALIFY ROW_NUMBER() OVER (PARTITION BY service_id, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, COUNT(*) as total,
    MIN(start_date) as min_date,
    MAX(end_date) as max_date
FROM gtfs_silver.calendar
GROUP BY city
ORDER BY city

### Calendar Dates

In [0]:
CREATE OR REPLACE TABLE gtfs_silver.calendar_dates AS
SELECT service_id,
    TO_DATE(date, 'yyyyMMdd') AS date,
    CAST(exception_type AS int) AS exception_type,
    city,
    ingestion_ts
FROM gtfs_bronze.calendar_dates
QUALIFY ROW_NUMBER() OVER (PARTITION BY service_id, date, city ORDER BY ingestion_ts DESC) = 1;

SELECT city, exception_type, COUNT(*) as total
FROM gtfs_silver.calendar_dates
GROUP BY city, exception_type
ORDER BY city, exception_type